In [ ]:
# ============================================================
# STAGE 1 — SETUP + OPEN PDF
# One complete executable cell
# ============================================================

!pip install -q docling google-genai faiss-cpu pypdf pillow tqdm

import os
import json
import re
import time
import random
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from pypdf import PdfReader

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

from google import genai
from google.genai import types

import faiss

# ------------------------------------------------------------
# GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/Complex_PDF_RAG")

SOURCE_DIR = DRIVE_ROOT / "source"
DOCLING_DIR = DRIVE_ROOT / "docling"
IMAGE_DIR = DRIVE_ROOT / "images"
GEMINI_DIR = DRIVE_ROOT / "gemini"
CANONICAL_DIR = DRIVE_ROOT / "canonical"
CHUNK_DIR = DRIVE_ROOT / "chunks"
VECTOR_DIR = DRIVE_ROOT / "vectors"

for directory in [
    DRIVE_ROOT, SOURCE_DIR, DOCLING_DIR, IMAGE_DIR,
    GEMINI_DIR, CANONICAL_DIR, CHUNK_DIR, VECTOR_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# INPUT PDF
# Change this path only if your source PDF has another name/path.
# ------------------------------------------------------------

PDF_PATH = Path("/content/Annu_Projects_Important_20_Pages.pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

DRIVE_PDF_PATH = SOURCE_DIR / PDF_PATH.name

if not DRIVE_PDF_PATH.exists():
    shutil.copy2(PDF_PATH, DRIVE_PDF_PATH)

PDF_PATH = DRIVE_PDF_PATH

reader = PdfReader(str(PDF_PATH))
TOTAL_PAGES = len(reader.pages)

# ------------------------------------------------------------
# EMBEDDING CONFIG
# ------------------------------------------------------------

EMBEDDING_MODEL_NAME = "gemini-embedding-2"
EMBEDDING_OUTPUT_DIMENSIONALITY = 768

# Gemini free-tier controls from the current AI Studio project.
# Keep these configurable because quotas can change by project/tier.
GEMINI_EMBEDDING_RPM_LIMIT = 100
GEMINI_EMBEDDING_TPM_LIMIT = 30_000
GEMINI_EMBEDDING_RPD_LIMIT = 1_000

# We intentionally leave safety headroom below the hard limits.
GEMINI_EMBEDDING_SAFE_RPM = 90
GEMINI_EMBEDDING_DAILY_LIMIT = 950
GEMINI_EMBEDDING_BATCH_SIZE = 50
GEMINI_EMBEDDING_INTER_BATCH_DELAY = 35
GEMINI_EMBEDDING_MIN_REQUEST_INTERVAL = 0.75
GEMINI_EMBEDDING_MAX_RETRIES = 6
GEMINI_EMBEDDING_CHECKPOINT_EVERY = 50

# ------------------------------------------------------------
# DOCLING CONFIG
# ------------------------------------------------------------

pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.generate_picture_images = True
pipeline_options.do_picture_description = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

print("=" * 80)
print("STAGE 1 COMPLETE — SETUP")
print("=" * 80)
print("PDF       :", PDF_PATH)
print("Pages     :", TOTAL_PAGES)
print("Project   :", DRIVE_ROOT)
print("Embedding :", EMBEDDING_MODEL_NAME)
print("Dimensions:", EMBEDDING_OUTPUT_DIMENSIONALITY)
print("Tables    : ON")
print("Pictures  : ON")
print("Gemini picture description in Docling: OFF")


Mounted at /content/drive
STAGE 1 COMPLETE — SETUP
PDF       : /content/drive/MyDrive/Complex_PDF_RAG/source/Annu_Projects_Important_20_Pages.pdf
Pages     : 20
Project   : /content/drive/MyDrive/Complex_PDF_RAG
Embedding : BAAI/bge-m3
Tables    : ON
Pictures  : ON
Gemini picture description in Docling: OFF


In [ ]:
# ============================================================
# STAGE 2 — DOCLING INGESTION
# Text + Tables + Images
# ONE COMPLETE EXECUTABLE CELL
# ============================================================

print("=" * 80)
print("STAGE 2 — DOCLING INGESTION")
print("=" * 80)

start_time = time.time()

# ------------------------------------------------------------
# 1. DOCILING PDF CONVERSION
# ------------------------------------------------------------

result = converter.convert(str(PDF_PATH))
doc = result.document

print(
    f"Docling conversion time: "
    f"{(time.time() - start_time) / 60:.2f} minutes"
)

# ------------------------------------------------------------
# 2. SAVE DOCLING DOCUMENT JSON
# ------------------------------------------------------------

DOCLING_JSON_PATH = DOCLING_DIR / "docling_document.json"

docling_data = result.document.export_to_dict()

with open(
    DOCLING_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        docling_data,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 3. UNIVERSAL OBJECT ACCESS
#
# Docling iterate_items() returns Pydantic objects.
# Older/custom code may sometimes deal with dictionaries.
# These helpers support both.
# ------------------------------------------------------------

def get_attr_or_key(obj, name, default=None):

    # Dictionary
    if isinstance(obj, dict):
        return obj.get(name, default)

    # Pydantic / normal object
    try:
        return getattr(obj, name, default)
    except Exception:
        return default


def object_to_dict(obj):

    if isinstance(obj, dict):
        return obj

    # Pydantic v2
    if hasattr(obj, "model_dump"):
        try:
            return obj.model_dump(
                mode="python"
            )
        except Exception:
            pass

    # Pydantic v1
    if hasattr(obj, "dict"):
        try:
            return obj.dict()
        except Exception:
            pass

    return {}


def clean_text(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace(
        "\xa0",
        " "
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


# ------------------------------------------------------------
# 4. TEXT EXTRACTION
# ------------------------------------------------------------

def get_item_text(item):

    # --------------------------------------------------------
    # Docling TextItem normally exposes .text
    # --------------------------------------------------------

    text = get_attr_or_key(
        item,
        "text",
        None
    )

    if isinstance(text, str) and text.strip():
        return clean_text(text)

    # --------------------------------------------------------
    # Some structures may expose .orig
    # --------------------------------------------------------

    orig = get_attr_or_key(
        item,
        "orig",
        None
    )

    if isinstance(orig, str) and orig.strip():
        return clean_text(orig)

    # --------------------------------------------------------
    # Dictionary fallback
    # --------------------------------------------------------

    if isinstance(item, dict):

        for key in [
            "text",
            "orig",
            "content"
        ]:

            value = item.get(key)

            if (
                isinstance(value, str)
                and value.strip()
            ):
                return clean_text(value)

    return ""


# ------------------------------------------------------------
# 5. PAGE NUMBER
# ------------------------------------------------------------

def get_page_number(item):

    prov = get_attr_or_key(
        item,
        "prov",
        []
    )

    if not prov:
        return None

    try:

        first_prov = prov[0]

        page_no = get_attr_or_key(
            first_prov,
            "page_no",
            None
        )

        if page_no is not None:
            return page_no

    except Exception:
        pass

    return None


# ------------------------------------------------------------
# 6. DOCUMENT ITEM COUNTS
# ------------------------------------------------------------

item_count = 0
text_count = 0
table_count = 0
picture_count = 0

for item, level in doc.iterate_items():

    item_count += 1

    item_type = type(item).__name__.lower()

    if "text" in item_type:
        text_count += 1

    elif "table" in item_type:
        table_count += 1

    elif "picture" in item_type:
        picture_count += 1


print("\nExtraction summary")
print("Total items :", item_count)
print("Text items  :", text_count)
print("Tables      :", table_count)
print("Pictures    :", picture_count)


# ------------------------------------------------------------
# 7. TABLE EXTRACTION
# ------------------------------------------------------------

def extract_table_cells(table):

    cells = []

    # --------------------------------------------------------
    # Docling TableItem
    # --------------------------------------------------------

    try:

        dataframe = table.export_to_dataframe()

        if dataframe is not None:

            for row_idx, row in dataframe.iterrows():

                for col_idx, value in enumerate(row):

                    if pd.isna(value):
                        value = ""

                    cells.append({
                        "row": int(row_idx),
                        "col": int(col_idx),
                        "text": clean_text(value)
                    })

            if cells:
                return cells

    except Exception:
        pass

    # --------------------------------------------------------
    # Dictionary fallback
    # --------------------------------------------------------

    table_dict = object_to_dict(table)

    data = table_dict.get(
        "data",
        {}
    )

    if isinstance(data, dict):

        grid = data.get(
            "grid",
            []
        )

        if isinstance(grid, list):

            for row_idx, row in enumerate(grid):

                if not isinstance(row, list):
                    continue

                for col_idx, cell in enumerate(row):

                    if isinstance(cell, dict):

                        value = (
                            cell.get("text")
                            or cell.get("value")
                            or ""
                        )

                    else:
                        value = cell

                    cells.append({
                        "row": row_idx,
                        "col": col_idx,
                        "text": clean_text(value)
                    })

    return cells


def table_to_text(table):

    # --------------------------------------------------------
    # FIRST: Docling's dataframe representation
    # --------------------------------------------------------

    try:

        dataframe = table.export_to_dataframe()

        if dataframe is not None:

            dataframe = dataframe.fillna("")

            lines = []

            # Header
            columns = [
                clean_text(x)
                for x in dataframe.columns
            ]

            if any(columns):
                lines.append(
                    " | ".join(columns)
                )

            # Rows
            for _, row in dataframe.iterrows():

                values = [
                    clean_text(value)
                    for value in row.tolist()
                ]

                if any(values):
                    lines.append(
                        " | ".join(
                            value if value else "-"
                            for value in values
                        )
                    )

            result_text = "\n".join(lines).strip()

            if result_text:
                return result_text

    except Exception:
        pass

    # --------------------------------------------------------
    # FALLBACK: GRID
    # --------------------------------------------------------

    cells = extract_table_cells(table)

    if not cells:

        return get_item_text(table)

    max_row = max(
        x["row"]
        for x in cells
    )

    max_col = max(
        x["col"]
        for x in cells
    )

    matrix = [
        ["" for _ in range(max_col + 1)]
        for _ in range(max_row + 1)
    ]

    for cell in cells:

        matrix[
            cell["row"]
        ][
            cell["col"]
        ] = cell["text"]

    lines = []

    for row in matrix:

        if not any(
            cell.strip()
            for cell in row
        ):
            continue

        lines.append(
            " | ".join(
                cell.strip()
                if cell.strip()
                else "-"
                for cell in row
            )
        )

    return "\n".join(lines)


# ------------------------------------------------------------
# 8. EXTRACT ALL IMAGES
# ------------------------------------------------------------

PICTURE_MANIFEST_PATH = (
    IMAGE_DIR / "picture_manifest.json"
)

picture_manifest = []

for idx, picture in enumerate(
    tqdm(
        doc.pictures,
        desc="Extracting pictures"
    ),
    start=1
):

    picture_id = f"picture_{idx:03d}"

    image_path = (
        IMAGE_DIR /
        f"{picture_id}.png"
    )

    try:

        # ----------------------------------------------------
        # Docling PictureItem -> PIL image
        # ----------------------------------------------------

        image = picture.get_image(doc)

        if image is None:
            continue

        image.save(
            image_path
        )

        # ----------------------------------------------------
        # PAGE + BBOX
        # ----------------------------------------------------

        page_no = None
        bbox = None

        try:

            prov = get_attr_or_key(
                picture,
                "prov",
                []
            )

            if prov:

                first_prov = prov[0]

                page_no = get_attr_or_key(
                    first_prov,
                    "page_no",
                    None
                )

                b = get_attr_or_key(
                    first_prov,
                    "bbox",
                    None
                )

                if b is not None:

                    bbox = {
                        "l": float(
                            get_attr_or_key(
                                b,
                                "l",
                                0
                            )
                        ),
                        "t": float(
                            get_attr_or_key(
                                b,
                                "t",
                                0
                            )
                        ),
                        "r": float(
                            get_attr_or_key(
                                b,
                                "r",
                                0
                            )
                        ),
                        "b": float(
                            get_attr_or_key(
                                b,
                                "b",
                                0
                            )
                        )
                    }

        except Exception:
            pass

        # ----------------------------------------------------
        # IMAGE SIZE
        # ----------------------------------------------------

        if bbox:

            bbox_width = abs(
                bbox["r"] - bbox["l"]
            )

            bbox_height = abs(
                bbox["b"] - bbox["t"]
            )

        else:

            bbox_width = image.width
            bbox_height = image.height

        picture_manifest.append({

            "picture_id": picture_id,

            "index": idx - 1,

            "page": page_no,

            "image_path": str(
                image_path
            ),

            "bbox": bbox,

            "width": image.width,

            "height": image.height,

            "bbox_width": bbox_width,

            "bbox_height": bbox_height
        })

    except Exception as e:

        print(
            f"Error extracting "
            f"{picture_id}: {e}"
        )


# ------------------------------------------------------------
# SAVE PICTURE MANIFEST
# ------------------------------------------------------------

with open(
    PICTURE_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        picture_manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 9. BUILD PAGE TEXT
# ------------------------------------------------------------

page_text = defaultdict(list)

for item, level in doc.iterate_items():

    text = get_item_text(item)

    if not text:
        continue

    page_no = get_page_number(
        item
    )

    if page_no is not None:

        page_text[
            page_no
        ].append(text)


# ------------------------------------------------------------
# 10. ADD NEARBY TEXT CONTEXT TO IMAGES
# ------------------------------------------------------------

picture_context_manifest = []

for picture_info in tqdm(
    picture_manifest,
    desc="Building picture context"
):

    page_no = picture_info["page"]

    nearby_text = []

    if page_no is not None:

        # Previous page
        nearby_text.extend(
            page_text.get(
                page_no - 1,
                []
            )
        )

        # Current page
        nearby_text.extend(
            page_text.get(
                page_no,
                []
            )
        )

        # Next page
        nearby_text.extend(
            page_text.get(
                page_no + 1,
                []
            )
        )

    # --------------------------------------------------------
    # Remove duplicate nearby text
    # --------------------------------------------------------

    seen = set()
    cleaned_context = []

    for text in nearby_text:

        text = text.strip()

        if (
            text
            and text not in seen
        ):

            seen.add(text)
            cleaned_context.append(text)

    picture_context_manifest.append({

        **picture_info,

        "nearby_text": cleaned_context
    })


PICTURE_CONTEXT_PATH = (
    IMAGE_DIR /
    "picture_context_manifest.json"
)

with open(
    PICTURE_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        picture_context_manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 11. VISUAL FILTERING
# ------------------------------------------------------------

VISUAL_MIN_WIDTH = 200
VISUAL_MIN_HEIGHT = 80

visual_manifest = []

for record in picture_context_manifest:

    width = record.get(
        "bbox_width",
        record.get(
            "width",
            0
        )
    )

    height = record.get(
        "bbox_height",
        record.get(
            "height",
            0
        )
    )

    if (
        width < VISUAL_MIN_WIDTH
        or height < VISUAL_MIN_HEIGHT
    ):
        continue

    image_path = Path(
        record["image_path"]
    )

    if not image_path.exists():
        continue

    visual_manifest.append({

        "picture_id":
            record["picture_id"],

        "page":
            record["page"],

        "image_path":
            str(image_path),

        "width":
            round(width, 2),

        "height":
            round(height, 2),

        "area":
            round(
                width * height,
                2
            ),

        "nearby_text":
            record.get(
                "nearby_text",
                []
            )
    })


visual_manifest.sort(
    key=lambda x: (
        x["page"]
        if x["page"] is not None
        else 0,

        x["picture_id"]
    )
)


VISUAL_MANIFEST_PATH = (
    IMAGE_DIR /
    "visual_manifest.json"
)

with open(
    VISUAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        visual_manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 12. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STAGE 2 COMPLETE — INGESTION")
print("=" * 80)

print(
    "Docling JSON :",
    DOCLING_JSON_PATH
)

print(
    "Pictures     :",
    len(picture_manifest)
)

print(
    "Visuals kept :",
    len(visual_manifest)
)

print(
    "Picture list :",
    PICTURE_MANIFEST_PATH
)

print(
    "Visual list  :",
    VISUAL_MANIFEST_PATH
)

STAGE 2 — DOCLING INGESTION
Docling conversion time: 0.81 minutes

Extraction summary
Total items : 217
Text items  : 125
Tables      : 23
Pictures    : 9


Extracting pictures:   0%|          | 0/9 [00:00<?, ?it/s]

Building picture context:   0%|          | 0/9 [00:00<?, ?it/s]


STAGE 2 COMPLETE — INGESTION
Docling JSON : /content/drive/MyDrive/Complex_PDF_RAG/docling/docling_document.json
Pictures     : 9
Visuals kept : 9
Picture list : /content/drive/MyDrive/Complex_PDF_RAG/images/picture_manifest.json
Visual list  : /content/drive/MyDrive/Complex_PDF_RAG/images/visual_manifest.json


In [ ]:
# ============================================================
# STAGE 3 — GEMINI VISUAL UNDERSTANDING
# Image -> Gemini -> searchable visual description
# Retry + fallback + checkpointing
# One complete executable cell
# ============================================================

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found in Colab Secrets."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

GEMINI_MODELS = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.5-flash",
    "gemini-3.7-flash"
]

MAX_RETRIES = 5
DELAY_BETWEEN_REQUESTS = 3
INITIAL_RETRY_DELAY = 2
MAX_RETRY_DELAY = 30

GEMINI_RESULTS_PATH = GEMINI_DIR / "visual_descriptions.json"

if GEMINI_RESULTS_PATH.exists():
    with open(GEMINI_RESULTS_PATH, "r", encoding="utf-8") as f:
        gemini_results = json.load(f)
else:
    gemini_results = []

results_by_picture = {
    item["picture_id"]: item
    for item in gemini_results
}


def build_visual_prompt(page_number):
    return f"""
You are analyzing a visual extracted from page {page_number}
of a complex PDF document for a Retrieval-Augmented Generation
(RAG) system.

Create a highly accurate, self-contained description of the visual.

Include, when visible:
1. Chart or figure title.
2. Type of visual.
3. Important categories, labels and legends.
4. Important numerical values.
5. Dates, years or periods.
6. Units.
7. Trends, comparisons and relationships.
8. Forecast values, if present.
9. Important annotations.
10. Source, if visible.
11. Meaning or interpretation directly supported by the visual.

Do NOT invent information that is not visible.
Do NOT provide opinions or recommendations.
Preserve exact terminology, names, labels and numerical values
whenever they are readable.

Return ONLY the description.
""".strip()


def classify_gemini_error(error):
    text = str(error).upper()

    if any(x in text for x in [
        "401", "403", "UNAUTHENTICATED",
        "PERMISSION_DENIED", "INVALID API KEY"
    ]):
        return "authentication"

    if any(x in text for x in [
        "429", "RESOURCE_EXHAUSTED",
        "RATE LIMIT", "QUOTA", "TOO MANY REQUESTS"
    ]):
        return "quota"

    if any(x in text for x in [
        "503", "UNAVAILABLE",
        "SERVICE UNAVAILABLE", "SERVER BUSY",
        "INTERNAL", "500", "502", "504"
    ]):
        return "temporary"

    return "permanent"


def call_gemini_with_retry(image_path, page_number, model_name):
    prompt = build_visual_prompt(page_number)

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=[
                    types.Part.from_bytes(
                        data=image_bytes,
                        mime_type="image/png"
                    ),
                    prompt
                ]
            )

            if not response.text:
                raise ValueError(
                    "Gemini returned an empty response."
                )

            return response.text.strip(), "success"

        except Exception as e:
            error_type = classify_gemini_error(e)

            print(
                f"{model_name} | "
                f"attempt {attempt + 1}/{MAX_RETRIES} | "
                f"{error_type} | {str(e)[:180]}"
            )

            if error_type == "authentication":
                return None, "authentication"

            if error_type == "permanent":
                return None, "permanent"

            if attempt < MAX_RETRIES - 1:
                delay = min(
                    INITIAL_RETRY_DELAY * (2 ** attempt)
                    + random.uniform(0, 1),
                    MAX_RETRY_DELAY
                )
                time.sleep(delay)

    return None, error_type


def process_visual(item):
    picture_id = item["picture_id"]
    page_number = item["page"]
    image_path = Path(item["image_path"])

    existing = results_by_picture.get(picture_id)

    if existing and existing.get("description"):
        print(f"{picture_id} -> already processed")
        return existing

    if not image_path.exists():
        return {
            "picture_id": picture_id,
            "page": page_number,
            "image_path": str(image_path),
            "description": None,
            "status": "image_missing"
        }

    for model_name in GEMINI_MODELS:
        description, status = call_gemini_with_retry(
            image_path=image_path,
            page_number=page_number,
            model_name=model_name
        )

        if description:
            return {
                "picture_id": picture_id,
                "page": page_number,
                "image_path": str(image_path),
                "model": model_name,
                "description": description,
                "status": "success"
            }

        if status == "authentication":
            return {
                "picture_id": picture_id,
                "page": page_number,
                "image_path": str(image_path),
                "description": None,
                "model": model_name,
                "status": "authentication_error"
            }

    return {
        "picture_id": picture_id,
        "page": page_number,
        "image_path": str(image_path),
        "description": None,
        "model": None,
        "status": "failed"
    }


def save_gemini_checkpoint():
    results = sorted(
        results_by_picture.values(),
        key=lambda x: (
            x.get("page", 0) or 0,
            x.get("picture_id", "")
        )
    )

    with open(GEMINI_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(
            results,
            f,
            ensure_ascii=False,
            indent=2
        )


print("=" * 80)
print("STAGE 3 — GEMINI VISUAL PROCESSING")
print("=" * 80)
print("Visual candidates:", len(visual_manifest))

for n, item in enumerate(visual_manifest, start=1):
    picture_id = item["picture_id"]

    existing = results_by_picture.get(picture_id)

    if existing and existing.get("description"):
        print(f"[{n}/{len(visual_manifest)}] {picture_id} -> cached")
        continue

    print(f"\n[{n}/{len(visual_manifest)}] Processing {picture_id}")

    result = process_visual(item)
    results_by_picture[picture_id] = result
    save_gemini_checkpoint()

    print("Status:", result["status"])

    if result["status"] == "authentication_error":
        print("Authentication error — stopping.")
        break

    if n < len(visual_manifest):
        time.sleep(DELAY_BETWEEN_REQUESTS)

save_gemini_checkpoint()

successful = sum(
    x.get("status") == "success"
    for x in results_by_picture.values()
)

print("\n" + "=" * 80)
print("STAGE 3 COMPLETE")
print("=" * 80)
print("Successful:", successful)
print("Total     :", len(results_by_picture))
print("Saved     :", GEMINI_RESULTS_PATH)


STAGE 3 — GEMINI VISUAL PROCESSING
Visual candidates: 9

[1/9] Processing picture_001
Status: success

[2/9] Processing picture_002
Status: success

[3/9] Processing picture_003
Status: success

[4/9] Processing picture_004
Status: success

[5/9] Processing picture_005
Status: success

[6/9] Processing picture_006
Status: success

[7/9] Processing picture_007
Status: success
[8/9] picture_008 -> cached
[9/9] picture_009 -> cached

STAGE 3 COMPLETE
Successful: 57
Total     : 57
Saved     : /content/drive/MyDrive/Complex_PDF_RAG/gemini/visual_descriptions.json


In [ ]:
# ============================================================
# STAGE 4 — CANONICAL DOCUMENT
# Combine TEXT + TABLE + GEMINI VISUAL DESCRIPTION
# One complete executable cell
# ============================================================

from collections import defaultdict

# ------------------------------------------------------------
# 1. RELOAD SAVED INGESTION DATA
# ------------------------------------------------------------

with open(DOCLING_JSON_PATH, "r", encoding="utf-8") as f:
    docling_data = json.load(f)

with open(GEMINI_RESULTS_PATH, "r", encoding="utf-8") as f:
    gemini_visuals = json.load(f)

# ------------------------------------------------------------
# 2. LOOKUPS
# ------------------------------------------------------------

texts_by_ref = {
    item.get("self_ref"): item
    for item in docling_data.get("texts", [])
    if item.get("self_ref")
}

tables_by_ref = {
    item.get("self_ref"): item
    for item in docling_data.get("tables", [])
    if item.get("self_ref")
}

pictures_by_ref = {
    item.get("self_ref"): item
    for item in docling_data.get("pictures", [])
    if item.get("self_ref")
}

groups_by_ref = {
    item.get("self_ref"): item
    for item in docling_data.get("groups", [])
    if item.get("self_ref")
}

gemini_by_picture = {
    item["picture_id"]: item
    for item in gemini_visuals
    if item.get("status") == "success"
}

picture_ref_to_id = {}

for idx, picture in enumerate(
    docling_data.get("pictures", []),
    start=1
):
    self_ref = picture.get("self_ref")

    if self_ref:
        picture_ref_to_id[self_ref] = f"picture_{idx:03d}"


def resolve_ref(ref):
    if isinstance(ref, dict):
        ref = ref.get("$ref")

    if not isinstance(ref, str):
        return None

    if ref in texts_by_ref:
        return "text", texts_by_ref[ref]

    if ref in tables_by_ref:
        return "table", tables_by_ref[ref]

    if ref in pictures_by_ref:
        return "picture", pictures_by_ref[ref]

    if ref in groups_by_ref:
        return "group", groups_by_ref[ref]

    return None


def get_page_info(item):
    prov = item.get("prov", [])

    if prov and isinstance(prov[0], dict):
        return prov[0].get("page_no")

    return None


# ------------------------------------------------------------
# 3. BUILD CANONICAL ELEMENTS
# ------------------------------------------------------------

canonical_elements = []

body = docling_data.get("body", {})
body_children = body.get("children", [])

for position, child in enumerate(body_children):

    resolved = resolve_ref(child)

    if resolved is None:
        continue

    item_type, item = resolved
    page_no = get_page_info(item)

    # --------------------------------------------------------
    # TEXT
    # --------------------------------------------------------

    if item_type == "text":
        text = get_item_text(item)

        if not text:
            continue

        canonical_elements.append({
            "element_id": f"element_{len(canonical_elements)+1:06d}",
            "type": "text",
            "position": position,
            "page": page_no,
            "text": text,
            "label": item.get("label", "text"),
            "source": {
                "docling_ref": item.get("self_ref")
            }
        })

    # --------------------------------------------------------
    # TABLE
    # --------------------------------------------------------

    elif item_type == "table":

        table_text = table_to_text(item)

        if not table_text:
            continue

        canonical_elements.append({
            "element_id": f"element_{len(canonical_elements)+1:06d}",
            "type": "table",
            "position": position,
            "page": page_no,
            "text": table_text,
            "content": table_text,
            "source": {
                "docling_ref": item.get("self_ref")
            }
        })

    # --------------------------------------------------------
    # PICTURE -> GEMINI DESCRIPTION
    # --------------------------------------------------------

    elif item_type == "picture":

        picture_ref = item.get("self_ref")
        picture_id = picture_ref_to_id.get(picture_ref)

        gemini_result = gemini_by_picture.get(picture_id)

        if not gemini_result:
            continue

        description = (
            gemini_result.get("description")
            or ""
        ).strip()

        if not description:
            continue

        canonical_elements.append({
            "element_id": f"element_{len(canonical_elements)+1:06d}",
            "type": "visual",
            "position": position,
            "page": page_no,
            "picture_id": picture_id,
            "image_path": gemini_result.get("image_path"),
            "visual_description": description,
            "text": description,
            "content": description,
            "source": {
                "docling_ref": picture_ref,
                "gemini_model": gemini_result.get("model")
            }
        })

# ------------------------------------------------------------
# 4. CREATE DOCUMENT
# ------------------------------------------------------------

canonical_document = {
    "schema_version": "1.0",
    "document_name": docling_data.get("name"),
    "source": {
        "type": "pdf",
        "origin": docling_data.get("origin")
    },
    "statistics": {
        "total_elements": len(canonical_elements),
        "text_elements": sum(
            x["type"] == "text"
            for x in canonical_elements
        ),
        "table_elements": sum(
            x["type"] == "table"
            for x in canonical_elements
        ),
        "visual_elements": sum(
            x["type"] == "visual"
            for x in canonical_elements
        )
    },
    "elements": canonical_elements
}

CANONICAL_JSON_PATH = (
    CANONICAL_DIR / "canonical_document.json"
)

with open(CANONICAL_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        canonical_document,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 5. REMOVE PAGE HEADERS/FOOTERS FOR CHUNKING
# ------------------------------------------------------------

REMOVE_LABELS = {
    "page_header",
    "page_footer"
}

chunking_elements = []

for element in canonical_elements:

    if (
        element["type"] == "text"
        and element.get("label", "text") in REMOVE_LABELS
    ):
        continue

    chunking_elements.append(element)

# ------------------------------------------------------------
# 6. QUALITY REPORT
# ------------------------------------------------------------

type_counts = Counter(
    x["type"]
    for x in chunking_elements
)

print("=" * 80)
print("STAGE 4 COMPLETE — CANONICAL DOCUMENT")
print("=" * 80)

print("Canonical file:", CANONICAL_JSON_PATH)
print("\nCanonical elements:")
print("  Text   :", type_counts.get("text", 0))
print("  Tables :", type_counts.get("table", 0))
print("  Visuals:", type_counts.get("visual", 0))
print("\nElements used for chunking:", len(chunking_elements))


STAGE 4 COMPLETE — CANONICAL DOCUMENT
Canonical file: /content/drive/MyDrive/Complex_PDF_RAG/canonical/canonical_document.json

Canonical elements:
  Text   : 125
  Tables : 23
  Visuals: 9

Elements used for chunking: 157


In [ ]:
# ============================================================
# STAGE 5 — STRUCTURE-AWARE CHUNKING
# TEXT -> section-aware chunks
# TABLE -> intact chunk
# VISUAL -> intact Gemini-description chunk
# One complete executable cell
# ============================================================

TARGET_CHARS = 1800
MAX_CHARS = 3200
MIN_STANDALONE_CHARS = 150

chunks = []

current_parts = []
current_element_ids = []
current_element_types = []
current_pages = []
current_section = ""


def normalize_spaces(text):
    text = text or ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_sentences(text):
    text = normalize_spaces(text)

    if not text:
        return []

    paragraphs = re.split(
        r"\n\s*\n",
        text
    )

    sentences = []

    for paragraph in paragraphs:
        paragraph = paragraph.strip()

        if not paragraph:
            continue

        parts = re.split(
            r"(?<=[.!?])\s+(?=[A-Z₹0-9(\[])",
            paragraph
        )

        for part in parts:
            part = part.strip()

            if part:
                sentences.append(part)

    return sentences


def is_probable_page_number(text):
    text = normalize_spaces(text)

    if not text:
        return False

    if re.fullmatch(r"\d{1,4}", text):
        return True

    if re.fullmatch(
        r"[ivxlcdmIVXLCDM]{1,8}",
        text
    ):
        return True

    return False


def is_low_value_fragment(text):
    text = normalize_spaces(text)

    if not text:
        return True

    if is_probable_page_number(text):
        return True

    return len(text) < MIN_STANDALONE_CHARS


def current_text():
    return "\n\n".join(
        x for x in current_parts if x.strip()
    ).strip()


def flush_chunk():
    global current_parts
    global current_element_ids
    global current_element_types
    global current_pages

    text = current_text()

    if not text:
        current_parts = []
        current_element_ids = []
        current_element_types = []
        current_pages = []
        return

    chunks.append({
        "chunk_id": f"chunk_{len(chunks):06d}",
        "page_start": min(current_pages) if current_pages else None,
        "page_end": max(current_pages) if current_pages else None,
        "section": current_section,
        "content": text,
        "element_ids": current_element_ids.copy(),
        "element_types": current_element_types.copy(),
        "has_table": "table" in current_element_types,
        "has_visual": "visual" in current_element_types
    })

    current_parts = []
    current_element_ids = []
    current_element_types = []
    current_pages = []


def add_element(element, content):
    content = normalize_spaces(content)

    if not content:
        return

    current_parts.append(content)
    current_element_ids.append(element["element_id"])
    current_element_types.append(element["type"])

    if element.get("page") is not None:
        current_pages.append(element["page"])


def add_text_sentence(element, sentence):
    global current_parts

    candidate = current_text()

    if candidate:
        candidate = candidate + "\n\n" + sentence
    else:
        candidate = sentence

    if len(candidate) <= TARGET_CHARS:
        add_element(element, sentence)
        return

    if current_text():
        flush_chunk()

        if len(sentence) <= MAX_CHARS:
            add_element(element, sentence)
            return

    # Sentence itself is too large.
    words = sentence.split()
    buffer = []

    for word in words:
        test = " ".join(buffer + [word])

        if len(test) <= MAX_CHARS:
            buffer.append(word)
        else:
            if buffer:
                add_element(
                    element,
                    " ".join(buffer)
                )
                flush_chunk()

            buffer = [word]

    if buffer:
        add_element(
            element,
            " ".join(buffer)
        )


# ------------------------------------------------------------
# MAIN CHUNKING
# ------------------------------------------------------------

for element in chunking_elements:

    element_type = element["type"]
    label = element.get("label", "")
    page = element.get("page")

    # --------------------------------------------------------
    # SECTION HEADER
    # --------------------------------------------------------

    if (
        element_type == "text"
        and label == "section_header"
    ):

        header = normalize_spaces(
            element.get("text", "")
        )

        if not header:
            continue

        flush_chunk()
        current_section = header

        # Keep section heading as retrieval content.
        add_element(
            element,
            header
        )

        continue

    # --------------------------------------------------------
    # TABLE
    # --------------------------------------------------------

    if element_type == "table":

        flush_chunk()

        table_text = normalize_spaces(
            element.get("content")
            or element.get("text")
            or ""
        )

        if table_text:
            add_element(
                element,
                table_text
            )
            flush_chunk()

        continue

    # --------------------------------------------------------
    # VISUAL
    # --------------------------------------------------------

    if element_type == "visual":

        flush_chunk()

        description = normalize_spaces(
            element.get("visual_description")
            or element.get("content")
            or element.get("text")
            or ""
        )

        if not description:
            continue

        visual_text = (
            "[VISUAL]\n"
            f"Picture ID: {element.get('picture_id')}\n"
            f"Page: {page}\n\n"
            f"{description}"
        )

        add_element(
            element,
            visual_text
        )

        flush_chunk()

        continue

    # --------------------------------------------------------
    # NORMAL TEXT
    # --------------------------------------------------------

    if element_type == "text":

        text = normalize_spaces(
            element.get("text")
            or element.get("content")
            or ""
        )

        if not text:
            continue

        if is_low_value_fragment(text):
            continue

        sentences = split_sentences(text)

        if not sentences:
            continue

        for sentence in sentences:
            add_text_sentence(
                element,
                normalize_spaces(sentence)
            )

flush_chunk()

# ------------------------------------------------------------
# MERGE VERY SMALL CHUNKS
# ------------------------------------------------------------

clean_chunks = []

for chunk in chunks:

    content = normalize_spaces(
        chunk["content"]
    )

    if len(content) >= MIN_STANDALONE_CHARS:
        chunk["content"] = content
        clean_chunks.append(chunk)
        continue

    if clean_chunks:

        previous = clean_chunks[-1]

        merged = (
            previous["content"]
            + "\n\n"
            + content
        )

        if len(merged) <= MAX_CHARS:
            previous["content"] = merged

            previous["page_end"] = max(
                previous["page_end"] or 0,
                chunk["page_end"] or 0
            )

            previous["element_ids"].extend(
                chunk["element_ids"]
            )

            previous["element_types"].extend(
                chunk["element_types"]
            )

            previous["has_table"] = (
                previous["has_table"]
                or chunk["has_table"]
            )

            previous["has_visual"] = (
                previous["has_visual"]
                or chunk["has_visual"]
            )

            continue

    chunk["content"] = content
    clean_chunks.append(chunk)

chunks = clean_chunks

for i, chunk in enumerate(chunks):
    chunk["chunk_id"] = f"chunk_{i:06d}"

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

CHUNKS_FINAL_V2_PATH = (
    CHUNK_DIR / "chunks_final_v2.json"
)

with open(
    CHUNKS_FINAL_V2_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# QUALITY CHECK
# ------------------------------------------------------------

sizes = [
    len(x["content"])
    for x in chunks
]

print("=" * 80)
print("STAGE 5 COMPLETE — CHUNKING")
print("=" * 80)
print("Total chunks       :", len(chunks))

if sizes:
    print("Average characters :", round(sum(sizes) / len(sizes)))
    print("Minimum characters :", min(sizes))
    print("Maximum characters :", max(sizes))

print(
    "Chunks with tables :",
    sum(x["has_table"] for x in chunks)
)

print(
    "Chunks with visuals:",
    sum(x["has_visual"] for x in chunks)
)

print("Saved:", CHUNKS_FINAL_V2_PATH)


STAGE 5 COMPLETE — CHUNKING
Total chunks       : 62
Average characters : 923
Minimum characters : 116
Maximum characters : 3499
Chunks with tables : 21
Chunks with visuals: 9
Saved: /content/drive/MyDrive/Complex_PDF_RAG/chunks/chunks_final_v2.json


In [ ]:
# ============================================================
# STAGE 6 — GEMINI EMBEDDING 2 API + FAISS
# Quota-safe ingestion for ~2K+ chunks with checkpointing
# ============================================================

CHUNKS_FINAL_V2_PATH = CHUNK_DIR / "chunks_final_v2.json"
EMBEDDINGS_PATH = VECTOR_DIR / "embeddings.npy"
METADATA_PATH = VECTOR_DIR / "embedding_metadata.json"
FAISS_INDEX_PATH = VECTOR_DIR / "faiss.index"
EMBEDDING_CHECKPOINT_PATH = VECTOR_DIR / "gemini_embedding_checkpoint.json"

# ------------------------------------------------------------
# 1. LOAD CHUNKS
# ------------------------------------------------------------

with open(CHUNKS_FINAL_V2_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

if not chunks:
    raise ValueError("No chunks found.")

# ------------------------------------------------------------
# 2. GEMINI CLIENT
# ------------------------------------------------------------

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not found in Colab Secrets.")

client = genai.Client(api_key=GEMINI_API_KEY)

print("=" * 80)
print("STAGE 6 — GEMINI EMBEDDING 2")
print("=" * 80)
print("Model             :", EMBEDDING_MODEL_NAME)
print("Dimensions        :", EMBEDDING_OUTPUT_DIMENSIONALITY)
print("Chunks            :", len(chunks))
print("Configured RPM    :", GEMINI_EMBEDDING_RPM_LIMIT)
print("Configured TPM    :", GEMINI_EMBEDDING_TPM_LIMIT)
print("Configured RPD    :", GEMINI_EMBEDDING_RPD_LIMIT)
print("Daily processing  :", GEMINI_EMBEDDING_DAILY_LIMIT)
print("Batch/checkpoint  :", GEMINI_EMBEDDING_BATCH_SIZE)

# ------------------------------------------------------------
# 3. RETRIEVAL-TASK FORMATTING
# Gemini Embedding 2 recommends an asymmetric format for retrieval:
# documents -> title + text; queries -> search-result task + query.
# ------------------------------------------------------------

def prepare_document(chunk):
    title = chunk.get("section") or "none"
    content = str(chunk.get("content", "")).strip()
    if not content:
        raise ValueError(f"Empty chunk content: {chunk.get('chunk_id')}")
    return f"title: {title} | text: {content}"

texts = [prepare_document(chunk) for chunk in chunks]

# ------------------------------------------------------------
# 4. SAFE RATE LIMITING + RETRIES
# ------------------------------------------------------------

last_request_time = 0.0

def wait_for_request_slot():
    global last_request_time
    elapsed = time.monotonic() - last_request_time
    if elapsed < GEMINI_EMBEDDING_MIN_REQUEST_INTERVAL:
        time.sleep(GEMINI_EMBEDDING_MIN_REQUEST_INTERVAL - elapsed)
    last_request_time = time.monotonic()

def embedding_error_is_retryable(exc):
    text = str(exc).upper()
    return any(x in text for x in [
        "429", "RESOURCE_EXHAUSTED", "RATE LIMIT", "TOO MANY REQUESTS",
        "500", "502", "503", "504", "UNAVAILABLE", "INTERNAL", "DEADLINE"
    ])

def embed_one(text, chunk_number):
    for attempt in range(1, GEMINI_EMBEDDING_MAX_RETRIES + 1):
        try:
            wait_for_request_slot()
            response = client.models.embed_content(
                model=EMBEDDING_MODEL_NAME,
                contents=text,
                config=types.EmbedContentConfig(
                    output_dimensionality=EMBEDDING_OUTPUT_DIMENSIONALITY
                )
            )
            if not response.embeddings:
                raise ValueError("Gemini returned no embedding.")
            values = np.asarray(response.embeddings[0].values, dtype="float32")
            if values.shape[0] != EMBEDDING_OUTPUT_DIMENSIONALITY:
                raise ValueError(
                    f"Unexpected embedding dimension {values.shape[0]} "
                    f"for chunk {chunk_number}."
                )
            if not np.isfinite(values).all():
                raise ValueError(f"NaN/Inf detected for chunk {chunk_number}.")
            return values
        except Exception as exc:
            print(
                f"Chunk {chunk_number} | attempt {attempt}/{GEMINI_EMBEDDING_MAX_RETRIES} | "
                f"{str(exc)[:220]}"
            )
            if not embedding_error_is_retryable(exc) or attempt == GEMINI_EMBEDDING_MAX_RETRIES:
                raise
            delay = min(2 ** (attempt - 1) + random.uniform(0, 1), 30)
            time.sleep(delay)

# ------------------------------------------------------------
# 5. CHECKPOINTED INGESTION
# Important: Gemini Embedding 2 Standard is free-tier; Batch API is
# currently not free-tier. Therefore this notebook uses quota-safe
# micro-batches (50 chunks) rather than paid Batch API jobs.
# ------------------------------------------------------------

if EMBEDDING_CHECKPOINT_PATH.exists():
    with open(EMBEDDING_CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)
    completed_indices = [int(x) for x in checkpoint.get("completed_indices", [])]
    embedding_map = {
        int(k): np.asarray(v, dtype="float32")
        for k, v in checkpoint.get("embedding_map", {}).items()
    }
else:
    completed_indices = []
    embedding_map = {}

completed_set = set(completed_indices)

# The free quota shown in your AI Studio project is 1,000 RPD.
# We therefore never intentionally submit more than 1,000 chunk requests/day.
remaining_budget_this_run = GEMINI_EMBEDDING_DAILY_LIMIT
indices_to_process = [
    i for i in range(len(chunks))
    if i not in completed_set
][:remaining_budget_this_run]

print("Already completed :", len(completed_indices))
print("Run request budget:", GEMINI_EMBEDDING_DAILY_LIMIT)
print("To process now    :", len(indices_to_process))
print("Remaining chunks  :", len(chunks) - len(completed_indices))

if not indices_to_process:
    print("No new embeddings required in this run/day.")
else:
    start_time = time.time()

    for batch_start in range(0, len(indices_to_process), GEMINI_EMBEDDING_BATCH_SIZE):
        batch_indices = indices_to_process[
            batch_start:batch_start + GEMINI_EMBEDDING_BATCH_SIZE
        ]

        print(
            f"\nBatch {batch_start // GEMINI_EMBEDDING_BATCH_SIZE + 1} | "
            f"chunks {batch_indices[0] + 1}-{batch_indices[-1] + 1} | "
            f"{len(batch_indices)} requests"
        )

        for idx in batch_indices:
            embedding_map[idx] = embed_one(texts[idx], idx + 1)
            completed_set.add(idx)

        # Checkpoint after every micro-batch so a Colab/Streamlit restart
        # does not lose completed embeddings.
        completed_indices = sorted(completed_set)
        with open(EMBEDDING_CHECKPOINT_PATH, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "model": EMBEDDING_MODEL_NAME,
                    "dimension": EMBEDDING_OUTPUT_DIMENSIONALITY,
                    "completed_indices": completed_indices,
                    "embedding_map": {str(k): v.tolist() for k, v in embedding_map.items()}
                },
                f
            )

        elapsed = time.time() - start_time
        processed = batch_start + len(batch_indices)
        rate = processed / elapsed if elapsed > 0 else 0
        eta = (len(indices_to_process) - processed) / rate if rate > 0 else 0
        print(
            f"Checkpoint saved | processed this run: {processed} | "
            f"rate: {rate:.2f} req/s | ETA: {eta / 60:.1f} min"
        )

        if batch_start + GEMINI_EMBEDDING_BATCH_SIZE < len(indices_to_process):
            time.sleep(GEMINI_EMBEDDING_INTER_BATCH_DELAY)

# ------------------------------------------------------------
# 6. BUILD COMPLETE EMBEDDING MATRIX IF ALL CHUNKS ARE READY
# ------------------------------------------------------------

if len(embedding_map) < len(chunks):
    print(
        f"\nPARTIAL INGESTION: {len(embedding_map)}/{len(chunks)} chunks embedded."
    )
    print(
        "Run this cell again after the RPD quota resets to continue from the checkpoint."
    )
    raise RuntimeError("Embedding quota reached before all chunks were processed.")

embeddings = np.vstack([embedding_map[i] for i in range(len(chunks))]).astype("float32")

assert embeddings.shape == (len(chunks), EMBEDDING_OUTPUT_DIMENSIONALITY)
assert np.isfinite(embeddings).all()

# Gemini Embedding 2 returns normalized embeddings; normalize again defensively.
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / np.clip(norms, 1e-12, None)

print("\nEmbedding shape :", embeddings.shape)
print("Embedding dtype :", embeddings.dtype)
print("Min norm        :", np.linalg.norm(embeddings, axis=1).min())
print("Max norm        :", np.linalg.norm(embeddings, axis=1).max())

# ------------------------------------------------------------
# 7. SAVE EMBEDDINGS
# ------------------------------------------------------------

np.save(EMBEDDINGS_PATH, embeddings)

# ------------------------------------------------------------
# 8. SAVE FAISS METADATA
# ------------------------------------------------------------

embedding_metadata = []
for i, chunk in enumerate(chunks):
    embedding_metadata.append({
        "faiss_index": i,
        "chunk_id": chunk["chunk_id"],
        "page_start": chunk["page_start"],
        "page_end": chunk["page_end"],
        "section": chunk["section"],
        "element_ids": chunk["element_ids"],
        "element_types": chunk["element_types"],
        "has_table": chunk["has_table"],
        "has_visual": chunk["has_visual"]
    })

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(embedding_metadata, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------
# 9. BUILD FAISS
# ------------------------------------------------------------

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

assert index.ntotal == len(embeddings)

faiss.write_index(index, str(FAISS_INDEX_PATH))
test_index = faiss.read_index(str(FAISS_INDEX_PATH))

assert test_index.ntotal == len(embeddings)
assert test_index.d == dimension

print("\n" + "=" * 80)
print("STAGE 6 COMPLETE — VECTOR DATABASE")
print("=" * 80)
print("Vectors    :", test_index.ntotal)
print("Dimensions :", test_index.d)
print("Metric     : Inner Product")
print("Normalized : YES")
print("FAISS file :", FAISS_INDEX_PATH)
print("Embeddings :", EMBEDDINGS_PATH)
print("Metadata   :", METADATA_PATH)


STAGE 6 — BGE-M3 EMBEDDINGS
Device: cuda
Model : BAAI/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Embedding time: 7.70 seconds

Embedding shape : (62, 1024)
Embedding dtype : float32
Min norm        : 0.99999994
Max norm        : 1.0000001

STAGE 6 COMPLETE — VECTOR DATABASE
Vectors    : 62
Dimensions : 1024
Metric     : Inner Product
Normalized : YES
FAISS file : /content/drive/MyDrive/Complex_PDF_RAG/vectors/faiss.index
Embeddings : /content/drive/MyDrive/Complex_PDF_RAG/vectors/embeddings.npy
Metadata   : /content/drive/MyDrive/Complex_PDF_RAG/vectors/embedding_metadata.json


In [ ]:
# ============================================================
# STAGE 7 — RAG QUERY + ANSWER GENERATION
# User question -> Gemini Embedding 2 -> FAISS -> context -> Gemini
# One complete executable cell
# ============================================================

# ------------------------------------------------------------
# 1. LOAD RAG ARTIFACTS
# ------------------------------------------------------------

with open(
    CHUNKS_FINAL_V2_PATH,
    "r",
    encoding="utf-8"
) as f:
    chunks = json.load(f)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:
    metadata = json.load(f)

index = faiss.read_index(
    str(FAISS_INDEX_PATH)
)

assert len(chunks) == len(metadata)
assert index.ntotal == len(chunks)

# ------------------------------------------------------------
# 2. GEMINI EMBEDDING 2 CLIENT
# ------------------------------------------------------------

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not found in Colab Secrets.")

client = genai.Client(api_key=GEMINI_API_KEY)

def prepare_query(question):
    return f"task: search result | query: {question}"

def embed_query(question):
    response = client.models.embed_content(
        model=EMBEDDING_MODEL_NAME,
        contents=prepare_query(question),
        config=types.EmbedContentConfig(
            output_dimensionality=EMBEDDING_OUTPUT_DIMENSIONALITY
        )
    )
    if not response.embeddings:
        raise ValueError("Gemini returned no query embedding.")
    vector = np.asarray(response.embeddings[0].values, dtype="float32")
    vector = vector / max(np.linalg.norm(vector), 1e-12)
    return vector.reshape(1, -1)

# ------------------------------------------------------------
# 3. GEMINI CLIENT
# ------------------------------------------------------------

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found in Colab Secrets."
    )

client = genai.Client(
    api_key=GEMINI_API_KEY
)

GEMINI_MODELS = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite"
]

# ------------------------------------------------------------
# 4. RETRIEVAL
# ------------------------------------------------------------

def retrieve_chunks(question, top_k=5):

    query_embedding = embed_query(question)

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        results.append({
            "rank": rank,
            "score": float(score),
            "chunk": chunks[idx],
            "metadata": metadata[idx]
        })

    return results


# ------------------------------------------------------------
# 5. BUILD CONTEXT
# ------------------------------------------------------------

def build_context(results):

    context_parts = []

    for item in results:

        chunk = item["chunk"]

        context_parts.append(
            f"""
SOURCE {item["rank"]}
Similarity score: {item["score"]:.4f}
Chunk ID: {chunk["chunk_id"]}
Page: {chunk["page_start"]} - {chunk["page_end"]}
Section: {chunk.get("section")}

CONTENT:
{chunk["content"]}
""".strip()
        )

    return "\n\n" + ("\n\n" + "=" * 60 + "\n\n").join(
        context_parts
    )


# ------------------------------------------------------------
# 6. GEMINI ERROR HANDLING
# ------------------------------------------------------------

def is_retryable_error(exc):

    error_text = str(exc).upper()

    retryable_codes = [
        "429",
        "500",
        "502",
        "503",
        "504",
        "UNAVAILABLE",
        "RESOURCE_EXHAUSTED",
        "INTERNAL",
        "DEADLINE"
    ]

    return any(
        code in error_text
        for code in retryable_codes
    )


def call_gemini_with_fallback(
    prompt,
    models=GEMINI_MODELS,
    max_retries=4,
    base_delay=3
):

    last_error = None

    for model_name in models:

        print(f"\nTrying Gemini model: {model_name}")

        for attempt in range(1, max_retries + 1):

            try:

                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt
                )

                if not response.text:
                    raise ValueError(
                        "Gemini returned an empty response."
                    )

                print(
                    f"SUCCESS -> {model_name}"
                )

                return (
                    response.text.strip(),
                    model_name
                )

            except Exception as exc:

                last_error = exc

                print(
                    f"Attempt {attempt}/{max_retries} failed: "
                    f"{str(exc)[:250]}"
                )

                if not is_retryable_error(exc):
                    break

                if attempt < max_retries:

                    delay = (
                        base_delay * (2 ** (attempt - 1))
                        + random.uniform(0, 2)
                    )

                    time.sleep(delay)

    raise RuntimeError(
        "All Gemini models failed."
    ) from last_error


# ------------------------------------------------------------
# 7. ANSWER GENERATION
# ------------------------------------------------------------

def generate_answer(question, results):

    context = build_context(results)

    prompt = f"""
You are a document question-answering assistant.

Answer the user's question using ONLY the retrieved
document context below.

IMPORTANT RULES:

1. Do not use outside knowledge.
2. Do not invent information.
3. Carefully compare all retrieved sources before answering.
4. Prefer the source that directly answers the exact question.
5. Do not select a source merely because it has the highest
   similarity score.
6. Pay close attention to:
   - dates
   - categories
   - units
   - metrics
   - geographic scope
   - time periods
7. Preserve numerical values from the document.
8. If calculation is required, calculate it using the
   numbers provided in the context.
9. If the retrieved context does not contain enough information,
   clearly say that the information is not available.
10. For multi-part questions, answer every part.
11. Mention the relevant page number when useful.
12. Do not mention FAISS, embeddings, chunks, or retrieval
    unless specifically asked.

FORMAT RULES:
- Return clean plain text or simple Markdown.
- Never use LaTeX.
- Use % for percentages.
- Use normal Markdown headings and bullet points when helpful.

RETRIEVED DOCUMENT CONTEXT
==========================

{context}

==========================

USER QUESTION
=============

{question}

==========================

FINAL ANSWER
============

Give a concise, accurate answer based strictly on the
retrieved document context.
"""

    return call_gemini_with_fallback(prompt)


# ------------------------------------------------------------
# 8. ASK QUESTION
# ------------------------------------------------------------

question = input(
    "\nEnter your question: "
).strip()

if not question:
    raise ValueError(
        "Question cannot be empty."
    )

TOP_K = 5

results = retrieve_chunks(
    question,
    top_k=TOP_K
)

# ------------------------------------------------------------
# 9. SHOW RETRIEVED CHUNKS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print(f"TOP {TOP_K} SEMANTICALLY RETRIEVED CHUNKS")
print("=" * 80)

for item in results:

    chunk = item["chunk"]

    print("\n" + "-" * 80)
    print("Rank       :", item["rank"])
    print("Score      :", f"{item['score']:.4f}")
    print("Chunk ID   :", chunk["chunk_id"])
    print(
        "Page       :",
        f"{chunk['page_start']} - {chunk['page_end']}"
    )
    print("Section    :", chunk.get("section"))
    print("Types      :", chunk.get("element_types"))
    print("Has table  :", chunk.get("has_table"))
    print("Has visual :", chunk.get("has_visual"))
    print("\nCONTENT:")
    print(chunk["content"])

# ------------------------------------------------------------
# 10. GENERATE FINAL ANSWER
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GENERATING FINAL ANSWER")
print("=" * 80)

answer, model_used = generate_answer(
    question,
    results
)

print("\n" + "=" * 80)
print("FINAL RAG ANSWER")
print("=" * 80)

print(answer)

print("\nModel used:", model_used)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Enter your question: Internet penetration increased from 36.70% in CY18 to 77.50% in CY25 and is expected to reach 86.00% by CY28. What is the total increase from CY18 to CY28 in percentage points?

TOP 5 SEMANTICALLY RETRIEVED CHUNKS

--------------------------------------------------------------------------------
Rank       : 1
Score      : 0.6901
Chunk ID   : chunk_000020
Page       : 6 - 6
Section    : 1.1.7 Internet Penetration
Types      : ['visual']
Has table  : False
Has visual : True

CONTENT:
[VISUAL]
Picture ID: picture_004
Page: 6

A line chart showing percentages from CY18 to CY28F. The data points and values for each period are as follows:
- CY18: 36.70%
- CY19: 45.80%
- CY20: 53.90%
- CY21: 60.60%
- CY22: 66.20%
- CY23: 70.90%
- CY24: 74.50%
- CY25: 77.50%
- CY26F: 81.00%
- CY27F: 83.50%
- CY28F: 86.00%

The values show a steady upward trend over time, with forecasts indicated for CY26F through CY28F.

--------------------------------------------------------------------